
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 임베딩과 AI Search

## 서론

검색 증강 생성(RAG) 시스템의 효과는 한 가지 핵심 요소, 즉 검색 파이프라인의 품질에 좌우됩니다. 관련 정보를 검색하기에 앞서, 먼저 비정형 텍스트를 *임베딩** 이라고 불리는 수치적 표현으로 변환하고 이를 특화된 **벡터 데이터베이스** 에 저장해야 합니다. 이번 단원에서는 임베딩 모델이 텍스트를 벡터로 변환하는 원리를 이해하는 것부터, 효율적인 검색을 위해 **벡터 유사도 알고리즘** 을 활용하는 방법까지 데이터 준비 생명주기 전반을 살펴봅니다. 또한, 결과의 품질을 대폭 향상시키는 하이브리드 검색 및 리랭킹(재순위화)과 같은 고도화된 검색 기술도 알아봅니다. 마지막으로, **Databricks AI Search** 가 Databricks 데이터 인텔리전스 플랫폼 내에서 이러한 구성 요소들을 어떻게 결합하여 안전한 서버리스 벡터 데이터베이스 솔루션을 제공하는지 확인해 봅니다.

## 수업 목표

본 단원을 마치면 다음을 수행할 수 있습니다.

* 임베딩 벡터의 주요 특징을 파악하고, 적절한 임베딩 모델을 선택하기 위한 기준을 평가할 수 있습니다.
* 유사도 검색, 전체 텍스트(Full-text) 검색, 하이브리드 검색 방법론을 비교하여 최적의 검색 전략을 결정할 수 있습니다.
* 운영 환경에서 정확한(Exact) 검색 알고리즘과 근사(Approximate) 검색 알고리즘 간의 절충 관계를 분석할 수 있습니다.
* 리랭킹이 어떻게 콘텍스트의 정확도를 높이고 RAG 애플리케이션의 환각(Hallucination) 현상을 줄이는지 설명할 수 있습니다.
* Databricks AI Search의 의 데이터 수집(Ingestion) 모드, 거버넌스 모델 및 아키텍처적 이점을 기술할 수 있습니다.

## A. 임베딩의 핵심 개념

이 장에서는 현대 정보 검색의 수학적 중추 역할을 하는 임베딩을 다루기 위해 꼭 필요한 기초 지식을 구축합니다. **비정형 텍스트가 어떻게 벡터 표현으로 변환되는지** 알아보고, 특정 도메인에 맞는 적절한 모델 선택이 왜 중요한지 살펴보며, 질의(query)와 문서(document) 공간 사이의 정렬이 갖는 핵심적인 중요성을 이해해 봅니다.

### A1. 임베딩 정의

임베딩은 일반적으로 딥러닝 모델에 의해 생성되는 내용의 수치적 표현입니다. 이 모델들은 고차원 비구조화 데이터(예: 텍스트)를 저차원 벡터, 즉 의미적 의미를 포착하는 부동소수점 배열로 변환합니다. 임베딩이 강력한 힘을 발휘하는 핵심 특성은 벡터 공간 내에서 유사한 개념들을 서로 가까이 매핑하는 능력에 있습니다. 관련성 높은 의미를 가진 단어나 구절들이 서로 가깝게 군집을 이루기 때문에, 정확한 키워드가 일치하지 않더라도 시스템이 개념적 관계를 식별할 수 있습니다.

### A2. 멀티모달 컨텍스트

**이번 과정에서는 비정형 텍스트를 중점적으로 다루지만**, 임베딩의 활용 범위가 단어에만 국한되지 않는다는 점을 짚고 넘어갈 필요가 있습니다. GPT-4o나 Gemini 1.5와 같은 멀티모달 모델은 이미지, 오디오, 텍스트를 하나의 통합된 벡터 공간으로 처리하고 임베딩할 수 있습니다. 이러한 기능 덕분에 교차 모달 검색(cross-modal retrieval)이 가능해집니다. 예컨대 텍스트로 검색해서 의미상 연관된 이미지를 찾거나, 글을 써서 오디오 콘텐츠를 검색하는 시나리오를 실현할 수 있습니다.

### A3. 임베딩 모델

임베딩 모델은 텍스트, 이미지, 오디오와 같은 고차원의 비정형 데이터를 저차원의 숫자 벡터로 변환하도록 설계된 특화된 머신러닝 모델(통상적으로 심층 신경망)입니다. **쉽게 말해 사람이 이해할 수 있는 콘텐츠를 컴퓨터가 읽을 수 있는 부동 소수점 숫자 목록으로 변환해 주는 번역기라고 볼 수 있습니다. 이 과정에서 의미가 유사한 입력 데이터들은 수학적으로 서로 가까운 위치의 벡터를 생성하게 됩니다.**

적절한 임베딩 모델을 선택하는 것은 검색 품질에 직접적인 영향을 미치는 중요한 아키텍처적 결정입니다. 이때 다음과 같은 핵심 요소를 고려해야 합니다.

- **어휘 크기 및 도메인:** 어떤 모델은 일반적인 웹 텍스트를 기반으로 학습하는 반면, 어떤 모델은 금융, 의료, 법률 문서 등 특정 도메인에 특화되어 있습니다. 전문적인 콘텐츠를 다룰 때는 도메인 특화 모델이 종종 더 우수한 결과를 보여줍니다.
- **컨텍스트 Window:** 모든 모델에는 입력 가능한 토큰의 최대 한도가 있습니다. 이 한도를 초과하는 텍스트는 잘리거나 무시되므로, 긴 문서를 다룰 때는 효과적인 청킹(chunking) 전략이 필수적입니다.
- **차원 수:** 차원이 높은 벡터(더 큰 배열)일수록 데이터의 미묘한 차이와 의미적 디테일을 더 잘 포착하지만, 그만큼 저장 비용과 검색 지연 시간(레이턴시)이 증가합니다. 따라서 필요한 정밀도와 운영상의 제약 조건 사이에서 균형을 맞추어야 합니다.

<!-- <img src="../Includes/images/03-vectorization.png" alt="Vectorization process illustrated" /> -->
![03-vectorization](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/03-vectorization.png)

*그림 1. 이 그림은 임베딩 모델이 데이터 청크를 처리하여 벡터를 생성하는 방식을 보여줍니다. 입력 데이터가 모델의 컨텍스트 Window 한도를 초과하면 초과 내용이 생략되어 결과 임베딩의 완성도에 영향을 줄 수 있습니다.*

### A4. 임베딩 정렬

검색이 효과적으로 작동하려면, 임베딩 모델이 원본 문서와 사용자 질의(쿼리)를 모두 동일한 벡터 공간 내에 표현해야 합니다. 만약 모델이 주로 긴 형태의 문서 위주로 학습되었는데 앱에서는 짧고 일상적인 질의를 사용한다면, 벡터 표현이 서로 잘 일치하지 않아 검색 결과가 떨어질 수 있습니다. 가장 좋은 해결책은 간단합니다. **문서를 인덱싱할 때와 질의를 처리할 때 동일한 임베딩 모델을 사용하는 것입니다.** 이렇게 하면 두 데이터가 동일한 수학적 공간에 존재하게 되어 의미 있는 비교가 가능해집니다.

## B. 벡터 저장 및 탐색 메커니즘

비정형 데이터를 임베딩으로 변환한 후에는, 고차원 벡터를 처리하고 효율적인 유사도 쿼리를 수행할 수 있는 전용 저장 공간이 필요합니다. 이번 장에서는 벡터 데이터베이스의 독특한 아키텍처를 살펴보고, 이것이 기존의 관계형 시스템과 어떻게 다른지 알아보겠습니다. 또한 대규모 환경에서 의미적으로 연관된 정보를 검색하는 데 사용되는 검색 알고리즘과 메트릭도 함께 살펴보겠습니다.

### B1. 벡터 데이터베이스의 역할

벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색할 수 있도록 특화된 데이터베이스입니다. 일치하는 데이터만을 찾는 기존의 전통적인 데이터베이스(SQL의 WHERE 절을 떠올려 보세요)와 달리, 벡터 데이터베이스는 완전히 동일하기보다는 개념적으로 서로 연관된 항목을 찾아내는 '유사도 검색'에 탁월한 성능을 발휘합니다. 또한, 생성·조회·수정·삭제(CRUD)와 같은 데이터베이스의 기본 기능을 그대로 유지하면서도, 벡터 연산에 최적화된 특화된 인덱싱 구조를 제공합니다.

### B2. 검색 방법

서로 다른 검색 방법은 서로 다른 검색 요구를 충족합니다:

- **유사도 검색 (Similarity Search):** 방식은 정확한 단어 일치 대신 의미론적 연관성을 기반으로 콘텐츠를 검색합니다. 이를 통해 "불안증에 대처하는 방법"과 같은 자연어 질의로도 "PTSD 대처법"이나 "스트레스 관리"처럼 다른 용어를 사용한 관련 결과를 찾아낼 수 있습니다.
- **전체 텍스트 검색 (Full-Text Search):** 이 전통적인 방식은 키워드 일치에 의존합니다. 부품 번호, 제품 코드, 고유 명사 같은 특정 용어를 찾는 데는 탁월하지만, 의미론적 의도를 파악하거나 동의어를 인식하지는 못합니다.
- **하이브리드 검색 (Hybrid Search):** 이 강력한 방식은 벡터 유사도 검색과 키워드 기반 검색을 결합한 것입니다. 의미론적 이해와 정확한 키워드 일치를 모두 활용함으로써, 하이브리드 검색은 일반적으로 두 방식 중 하나만 사용할 때보다 더 높은 검색 정확도를 제공합니다

### B3. 거리 및 유사도 지표

두 벡터가 얼마나 "유사한지" 판단하기 위해 우리는 크게 두 가지 유형의 지표, 즉 **거리 지표** 와 **유사도 지표** 를 사용하며, 각각의 지표는 서로 다른 검색 시나리오에 적합합니다. **거리 지표** 는 공간에서 두 벡터가 얼마나 떨어져 있는지 정량화할 때 사용되며, 클러스터링(군집화), 이상치 탐지 또는 벡터의 크기가 중요할 때 이상적입니다. 반면, **유사도 지표** 는 두 벡터의 방향이 얼마나 일치하는지 알고 싶을 때 사용되며, 크기보다 의미가 더 중요한 의미론적 검색, 문서 검색 및 대부분의 자연어 처리(NLP) 애플리케이션에 이상적입니다.

**거리 지표 (Distance Metrics)**  
- **유클리드 거리 (Euclidean Distance, L2):** 벡터 공간에서 두 점 사이의 직선거리를 측정합니다. 값이 작을수록 두 벡터가 더 유사함을 의미합니다. 클러스터링이나 이상 탐지처럼 모든 차원에서의 절대적인 차이를 고려해야 할 때 사용합니다.
- **맨해튼 거리 (Manhattan Distance, L1):** 모든 차원의 절대적 차이의 합을 구합니다. 값이 작을수록 두 벡터가 더 가깝다는 뜻입니다. 격자 형태의 데이터나 희소 데이터(sparse data)처럼 각 축을 따른 차이가 동일하게 중요할 때 유용합니다.

**유사도 지표 (Similarity Metrics)**  
- **코사인 유사도 (Cosine Similarity):** 두 벡터 사이의 각도에 대한 코사인 값을 측정합니다. 점수가 높을수록 유사도가 높음을 의미합니다. 벡터의 크기보다는 방향(의미론적 유사성)에 집중하기 때문에 텍스트 임베딩에서 가장 널리 쓰이는 지표이며, 문서의 길이나 크기 차이에 영향을 받지 않고 안정적인 결과를 제공합니다.


<!-- <img src="../Includes/images/03-vector-similarities.png" alt="Distance and similarity metrics" /> -->

![03-vector-similarities](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/03-vector-similarities.png)

*그림 2. 이 다이어그램은 위에 나열된 거리 지표와 유사성 지표를 시각화합니다*

### B4. 검색 전략

정확성과 성능 사이의 균형을 맞추기 위해 다음과 같은 두 가지 주요 전략이 사용됩니다.

- **K-Nearest Neighbors(KNN):** 쿼리 벡터와 데이터베이스 내 모든 벡터 간의 거리를 계산하는 정확한 탐색 방법입니다. 정확도는 매우 높지만 연산 비용이 많이 들고 대규모 데이터셋에는 적합하지 않습니다. 마치 내가 던진 질문을 수백만 개의 문서와 일일이 하나씩 대조하는 것과 같습니다.
- **Approximate Nearest Neighbors(ANN):** 약간의 정확도를 양보하는 대신 극적인 속도 향상을 얻는 전략입니다. ANN은 **HNSW(Hierarchical Navigable Small Worlds)** 나 **FAISS(Facebook AI Similarity Search)** 같은 정교한 인덱싱 알고리즘을 사용하여 벡터 공간을 효율적으로 탐색하며, 전체 중 일부 벡터만 확인하면서도 연관성이 매우 높은 결과를 찾아냅니다.

## C. 정밀도, 품질 및 재순위화(Reranking)

벡터 데이터베이스는 의미론적으로 유사한 콘텐츠를 찾는 강력한 메커니즘을 제공하지만, 몇 가지 한계점도 존재합니다. 이 섹션에서는 임베딩 품질의 미묘한 차이와 수학적 유사도와 실제 의미론적 연관성 사이에 존재할 수 있는 격차를 다룹니다. 또한, 검색 결과를 정제하고 언어 모델에 제공되는 컨텍스트의 정확도를 높이는 중요한 후처리 단계인 '재순위화(Reranking)'에 대해서도 소개합니다.

### C1. 임베딩 품질과 한계

여기서 중요한 통찰은 **'유사성'이 곧 '의미론적 연관성'을 뜻하지는 않는다는 점입니다.** 어떤 문서가 벡터 공간에서 사용자의 검색어와 수학적으로는 가까울 수 있지만, 실제로는 사실 관계가 맞지 않거나 맥락상 부적절할 수 있습니다. 임베딩의 품질은 모델, 학습 데이터, 그리고 사용자의 특정 도메인과 얼마나 잘 부합하는지에 따라 크게 좌우됩니다. 데이터 준비가 미흡하거나 모델의 학습 코퍼스와 애플리케이션 콘텐츠가 서로 일치하지 않으면, 검색 성능이 떨어지고 정보를 '유실'하는 결과로 이어질 수 있습니다.

또 다른 흔한 사례는 유사도 검색 결과 전체를 사용하는 대신 일부 문서만 선택하는 경우입니다. 토큰 제한이나 처리 비용 등의 이유로 문서 수를 제한해야 할 때는 가장 연관성이 높은 문서들이 상위에 노출되도록 해야 합니다. 바로 이 부분에서 리랭킹(재정렬)이 필수적인 역할을 합니다.

### C2. 리랭킹(재정렬) 과정

초기 검색의 정확도 격차를 줄이기 위해 파이프라인에 **reranker**를 추가합니다:

1. **초기 검색:** 벡터 저장소에서 빠른 ANN(근사 최근접 이웃) 알고리즘을 사용하여 광범위한 후보 문서 세트(통상 상위 20~50개)를 검색합니다.
1. **재순위 조정:** 특화된 모델(주로 크로스 인코더)이 각 후보 문서와 특정 쿼리 간의 관계를 상세히 분석하여 실제 연관성을 평가합니다.
1. **재정렬:** 리랭커의 연관성 점수를 바탕으로 문서를 다시 정렬하여, 언어 모델이 처리할 수 있도록 가장 관련성이 높은 정보를 최상단에 배치합니다.

<!-- <img src="../Includes/images/03-reranking.png" alt="Reranking process" width="500" /> -->

![03-reranking](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/03-reranking.png)

*그림 3. 이 도표는 리랭킹 과정이 어떻게 작동하는지 보여줍니다*

### C3. 장점 및 고려 사항(Trade-offs)

리랭킹(Reranking)을 도입할 때는 다음과 같은 중요 요소를 고려해야 합니다.

- **장점:** 리랭킹은 거대언어모델(LLM)에 제공되는 컨텍스트의 정확도를 대폭 향상시켜, 할루시네이션(환각 현상)을 직접적으로 줄이고 답변의 질을 높여줍니다. 초기 검색 결과를 정제함으로써 가장 관련성 높은 정보가 생성 단계에 전달되도록 보장합니다.
- **고려 사항:** 검색 파이프라인에 리랭커를 추가하면 대기 시간(지연 시간)과 비용이 모두 증가합니다. 리랭킹 모델이 쿼리와 후보 문서들을 실시간으로 처리해야 하므로 계산 오버헤드가 발생합니다. 따라서 특정 사용 사례에 맞춰 이러한 비용과 품질 향상 간의 균형을 조절해야 합니다.

## D. Databricks AI Search - 특징과 아키텍처

견고한 벡터 데이터베이스 인프라 구현은 복잡할 수 있지만, Databricks는 Databricks AI Search으로 이 과정을 단순화합니다. 이번 장에서는 해당 서비스의 아키텍처를 살펴보고, 데이터 자동 동기화를 위해 델타 레이크(Delta Lake)와 어떻게 매끄럽게 통합되는지 알아보겠습니다. 또한, 벡터 인덱스에 대한 안전하고 관리되는 접근을 보장하는 유니티 카탈로그(Unity Catalog) 기반의 통합 거버넌스 모델도 함께 살펴보겠습니다.

<!-- <img src="../Includes/images/03-vector-search-components.png" alt="Databricks AI Search components" /> -->
![03-vector-search-components](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/03-vector-search-components.png)

*그림 4. 이 도표는 Databricks AI Search*의 주요 구성 요소를 보여줍니다

### D1. 제품 개요

**Databricks AI Search**는 Databricks 레이크하우스에 직접 통합된 벡터 데이터베이스 솔루션입니다. 이 확장 가능한 저지연 서비스는 데이터의 벡터 표현을 메타데이터와 함께 저장하여, REST API 및 Python 클라이언트를 통한 실시간 유사도 검색을 지원합니다. RAG(검색 증강 생성) 애플리케이션의 정보 검색을 최적화하도록 맞춤 설계되었으며, 별도의 벡터 데이터베이스 인프라를 관리할 필요가 없습니다.

### D2. Delta 동기화 및 인덱싱

Databricks AI Search의 가장 강력한 특징 중 하나는 **Delta Lake**와의 긴밀한 통합입니다. **Delta Sync API**를 통해 Vector Index가 자동으로 소스 Delta 테이블과 동기화됩니다. 소스 테이블에 데이터를 추가, 업데이트 또는 삭제할 때 Vector Index가 자동으로 업데이트되어, 검색 시스템이 항상 최신 데이터를 반영하도록 수동 개입 없이 보장합니다. 이로 인해 임베딩을 소스 데이터와 동기화하는 운영 부담이 줄어듭니다.

### D3. 관리 및 인제스트 모드

Databricks AI Search는 임베딩을 수집하고 관리할 수 있도록 세 가지 유연한 방식을 제공하며, 사용자는 필요에 따라 제어 수준을 선택할 수 있습니다.

1. **관리형 임베딩 (Delta Sync):** 원시 텍스트가 포함된 소스 Delta 테이블을 제공하면 나머지 과정은 Databricks가 처리합니다. 시스템이 설정된 **Mosaic AI Model Serving** 엔드포인트(예: Foundation Model API)를 사용하여 임베딩을 자동으로 계산하고, 새로운 데이터를 처리하며, 인덱스를 업데이트합니다. 따라서 사용자가 임베딩 파이프라인을 직접 관리할 필요가 없습니다.

1. **자체 관리형 임베딩 (Delta Sync):** 사용자가 자체 커스텀 파이프라인을 사용하여 임베딩을 계산하고 이를 Delta 테이블에 저장합니다. AI Search 인덱스는 이 테이블과 동기화되어 사용자가 제공한 사전 계산된 벡터를 인덱싱합니다. 이 방식을 사용하면 자동 동기화의 이점을 누리면서도 임베딩 프로세스를 전적으로 제어할 수 있습니다.

1. **직접 액세스 CRUD API:** REST API 또는 Python SDK를 사용하여 AI Search 인덱스와 직접 상호작용할 수 있습니다. 이를 통해 기반이 되는 Delta 테이블 동기화에 의존하지 않고 벡터와 메타데이터를 직접 삽입, 업데이트 또는 삭제할 수 있어, 실시간 애플리케이션이나 커스텀 워크플로우에 이상적입니다.


<!-- <img src="../Includes/images/03-vector-search-managed-embeddings.png" alt="Databricks AI Search managed embeddings method" /> -->

![03-vector-search-managed-embeddings](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/03-vector-search-managed-embeddings.png)

*그림 5. 이 다이어그램은 AI Search이 자동 동기화를 통해 임베딩을 어떻게 관리하는지 보여줍니다.*

### D4. 거버넌스 및 접근 통제

Databricks AI Search는 **Unity Catalog**에 의해 관리되며, 데이터와 AI 자산 모두에 대해 **통합된 보안 모델을 제공합니다**. AI Search에서 생성된 인덱스는 Unity Catalog 내에서 보안 가능한 객체로 나타나, 관리자가 인덱스 수준에서 세분화된 접근 제어 목록(ACL)을 강제할 수 있게 합니다. 이로 인해 권한 있는 사용자와 애플리케이션만이 벡터 데이터를 쿼리하거나 수정할 수 있게 되어 전체 데이터 플랫폼에서 일관된 보안 정책을 유지할 수 있습니다.

## E. 요약

이번 단원에서는 RAG 시스템에서 데이터를 검색할 수 있도록 준비하는 전체 생명주기를 살펴보았습니다. 먼저 정형화되지 않은 텍스트와 기계가 읽을 수 있는 벡터를 연결하는 핵심 가교로써 **'임베딩'** 을 정의하고, 도메인과 쿼리 패턴의 특성에 맞춰 적절한 모델을 선택해야 함을 강조했습니다. 이어서 **벡터 데이터베이스** 의 작동 원리를 파악하며 정확한 검색(KNN)과 근사 검색(ANN) 전략의 차이점을 구분해 보았고, **하이브리드 검색** 과 **리랭킹(재순위화)** 을 통해 순수 벡터 유사도 검색의 한계를 극복하는 방법도 알아봤습니다. 마지막으로, **Delta Sync** 를 통해 임베딩 관리를 자동화하고 Unity Catalog와 연동하여 강력한 보안을 제공하는 **Databricks AI Search** 의 강력한 기능까지 함께 살펴보았습니다.

**핵심 요점:**

1. **임베딩과 정렬:** 임베딩은 벡터 공간에서 유사한 개념을 서로 가깝게 매핑하여 시멘틱(의미론적) 의미를 포착합니다. 효과적인 검색을 수행하려면 임베딩 모델이 문서와 쿼리 모두에 대해 공유 벡터 공간을 생성해야 합니다. 즉, 일치성을 보장하기 위해 두 과정에 반드시 동일한 모델을 사용해야 합니다.
2. **검색 정밀도:** **ANN** 알고리즘은 실제 운영 시스템에 필요한 속도와 확장성을 제공하지만, 노이즈를 필터링하고 언어 모델에 높은 관련성을 보장하기 위해 대개 **리랭킹** 단계를 추가하는 것이 필수적입니다. 이때 품질 향상 효과와 그에 따른 지연 시간 및 비용 증가 사이의 균형을 잘 맞추어야 합니다.
3. **통합 아키텍처:** **Databricks AI Search**는 **Delta Lake**와의 자동 동기화를 제공하고 다양한 수집 모드(관리형, 자체 관리형, 직접 CRUD)를 지원하여 운영을 단순화합니다. 이러한 통합 덕분에 별도의 벡터 데이터베이스 인프라를 관리하는 복잡성이 사라지며, Unity Catalog를 통해 엔터프라이즈급 거버넌스를 그대로 유지할 수 있습니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>